# CLIP-Anchored GAN (`f-CLSWGAN`) Training for Out-Of-Distribution Plant Disease Identification

This notebook trains the **CLIP-Anchored Feature-Generating GAN** (`train/train_feature_gen.py`, method `clip_feature_gen`) using the best configuration:
- **`config/vlm/v11.4_lora_compositional_config.json`** (*Best overall CLIP-Anchored GAN*: Compositional `[pair, crop, disease]` CLIP text conditioning on `v10.3` text-anchored LoRA image features — **61.85% Val HM**, **66.13% Test HM**, **90.05% Test Top-5 HM**).

### Pipeline Overview:
1. **Configuration & Parameters**: Experiment configurations (`CONFIG_PATHS`), Google Drive paths, and local SSD extraction paths.
2. **Google Drive Integration**: Mounts `/content/drive` to access the project folder (`Nat_AML_DPD_group_assignment`).
3. **Fast Local Extraction**: Copies `DPD_Small.7z` from Drive to Colab local SSD (`/content/`) and extracts with `7z` to maximize DataLoader throughput.
4. **Environment Setup**: Installs required Python dependencies (`open_clip_torch`, `timm`, etc.).
5. **Sequential Training & Drive Sync**: Restores any cached embeddings / prerequisite LoRA runs (`runs/vlm/v10.3`) from Google Drive (or trains `v10.3` automatically if missing), runs 2-stage `train.train_feature_gen` training, and syncs all outputs and caches back to Google Drive.
6. **Summary & Comparison**: Summarizes both **Uncalibrated ($\gamma=0$)** and **Calibrated ($\gamma$ from validation)** metrics side-by-side.


In [35]:
# ================= Step 1: Configuration & Experiment Parameters ================
import os
from pathlib import Path

# --- Model & Training Configurations (run sequentially in order) ---
# v11.4 is the best CLIP-anchored GAN configuration (compositional [pair, crop, disease]
# text condition + v10.3 text-anchored LoRA image features).
# Uncomment additional v11 configs below if you want to run the full ablation suite.
CONFIG_PATHS = [
    # "config/vlm/v14_clip_siglip_zeroshot_config.json",      # Untuned zero-shot SigLIP baseline (ViT-B-16-SigLIP-256)
    # "config/vlm/v14.1_clip_siglip_lora_config.json",        # LoRA + compositional prompts SigLIP (ViT-B-16-SigLIP-256)
    # "config/vlm/v11.4_lora_compositional_config.json",    # Best CLIP-anchored GAN (OpenAI CLIP + v10.3 LoRA)
    # "config/vlm/v11.3_lora_features_config.json",            # Pair-only condition on v10.3 LoRA features
    # "config/vlm/v11.1_compositional_condition_config.json",  # Compositional condition on frozen CLIP features
    # "config/vlm/v11_feature_gen_config.json",                # Base f-CLSWGAN on frozen CLIP features
]
if isinstance(CONFIG_PATHS, str):
    CONFIG_PATHS = [CONFIG_PATHS]

# Also run the Disease-Only SigLIP + Crop-Anchored GAN + ConvNeXt two-expert pipeline
# (both untuned --no-lora and LoRA-tuned for direct comparison).
RUN_SIGLIP_ANCHORED_GAN = True

# Prerequisite LoRA config used if a config specifies features.source == "lora_run"
# and runs/vlm/v10.3 is not yet present in Google Drive or local runs/.
PREREQ_LORA_CONFIGS = {
    "runs/vlm/v10.3": "config/vlm/v10.3_text_anchor_three_splits_config.json",
}

# --- Google Drive & Sync Paths ---
GDRIVE_PROJECT = Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment")
GDRIVE_DATASET_7Z = GDRIVE_PROJECT / "DPD_Small.7z"
GDRIVE_RUNS = GDRIVE_PROJECT / "runs"
SYNC_DIR = str(GDRIVE_RUNS)

# --- Fast Local Storage Paths (/content/) ---
LOCAL_ARCHIVE = Path("/content/DPD_Small.7z")
EXTRACT_DEST = Path("/content/dataset")

print("Configuration parameters loaded successfully:")
print(f"  Configs ({len(CONFIG_PATHS)}):")
for i, cfg_p in enumerate(CONFIG_PATHS, 1):
    print(f"    {i}. {cfg_p}")
print(f"  Project Dir:   {GDRIVE_PROJECT}")
print(f"  Sync Dir:      {SYNC_DIR}")


Configuration parameters loaded successfully:
  Configs (0):
  Project Dir:   /content/drive/MyDrive/Nat_AML_DPD_group_assignment
  Sync Dir:      /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs


In [36]:
# Step 2: Mount Google Drive & Verify Project Folder
from google.colab import drive
from pathlib import Path
import os

# Mount Drive
drive.mount('/content/drive')

# Verify paths configured in Step 1 exist
assert GDRIVE_PROJECT.exists(), f"Google Drive folder not found: {GDRIVE_PROJECT}"
assert GDRIVE_DATASET_7Z.exists(), f"Dataset archive not found at: {GDRIVE_DATASET_7Z}"
GDRIVE_RUNS.mkdir(parents=True, exist_ok=True)

print("Google Drive mounted and verified successfully!")
print(f"Project folder: {GDRIVE_PROJECT}")
print(f"Dataset archive: {GDRIVE_DATASET_7Z}")
print(f"Persistent runs folder: {GDRIVE_RUNS}")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Google Drive mounted and verified successfully!
Project folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment
Dataset archive: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/DPD_Small.7z
Persistent runs folder: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs


In [37]:
# Step 3: Extract Dataset to Fast Local Colab SSD (/content/)
import shutil
from pathlib import Path
import os

EXTRACT_DEST.mkdir(parents=True, exist_ok=True)

# 1. Copy .7z from Google Drive to local SSD (if not already copied)
if not LOCAL_ARCHIVE.exists():
    print(f"Copying {GDRIVE_DATASET_7Z} to local SSD ({LOCAL_ARCHIVE})...")
    shutil.copy2(GDRIVE_DATASET_7Z, LOCAL_ARCHIVE)
    print("Copy complete!")
else:
    print(f"Local archive {LOCAL_ARCHIVE} already present.")

# 2. Extract using 7z if CSVs are not yet present
def find_dpd_root(search_dir: Path) -> Path | None:
    for candidate in [search_dir, search_dir / "DPD_Small", *search_dir.glob("**/DPD_Small*")]:
        if candidate.is_dir() and (candidate / "DPD_Small_train.csv").exists():
            return candidate
    return None

dpd_root = find_dpd_root(EXTRACT_DEST)
if dpd_root is None:
    print("Extracting dataset with 7z (fast multithreaded)...", flush=True)
    !7z x -y /content/DPD_Small.7z -o/content/dataset
    dpd_root = find_dpd_root(EXTRACT_DEST)

assert dpd_root is not None, f"Could not find DPD_Small_train.csv inside {EXTRACT_DEST}!"

# 3. Verify all dataset components exist
assert (dpd_root / "DPD_Small_train.csv").exists(), f"Missing train CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_seen.csv").exists(), f"Missing test seen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_test_unseen.csv").exists(), f"Missing test unseen CSV in {dpd_root}"
assert (dpd_root / "DPD_Small_dataset").exists(), f"Missing DPD_Small_dataset folder in {dpd_root}"

# 4. Export DPD_ROOT environment variable (read by dpd/data.py)
os.environ["DPD_ROOT"] = str(dpd_root)
print(f"\nDataset verified successfully!")
print(f"DPD_ROOT set to: {os.environ['DPD_ROOT']}")


Local archive /content/DPD_Small.7z already present.

Dataset verified successfully!
DPD_ROOT set to: /content/dataset


In [38]:
# Step 4: Install Required Python Packages
!apt-get install -y -qq
!pip install -q open_clip_torch timm pandas pillow tqdm matplotlib
!nvidia-smi


Tue Nov 17 00:57:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA L4                      Off |   00000000:00:03.0 Off |                    0 |
| N/A   55C    P8             17W /   72W |       0MiB /  23034MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [44]:
# Step 5: Run CLIP-Anchored GAN Training Across All Configs
import json
import os
import shutil
import subprocess
import sys
import time
from pathlib import Path

# Automatically locate the directory that contains the 'train' and 'dpd' modules
candidates = [
    Path("/content"),
    Path.cwd(),
    Path("/content/Out-Of-Distribution-Plant-Disease-Identification"),
    Path("/content/drive/MyDrive/Nat_AML_DPD_group_assignment"),
]
project_root = None
for c in candidates:
    if (c / "train" / "train_feature_gen.py").exists() or (c / "train").is_dir():
        project_root = c.resolve()
        break

if project_root is None:
    matches = list(Path("/content").glob("**/train_feature_gen.py"))
    project_root = matches[0].parent.parent.resolve() if matches else Path("/content").resolve()

print(f"Project root located at: {project_root}")
os.chdir(str(project_root))

# Ensure PYTHONPATH contains project_root so 'train' and 'dpd' are always found
env = os.environ.copy()
env["PYTHONPATH"] = f"{str(project_root)}:{env.get('PYTHONPATH', '')}"

def resolve_config(rel_or_abs: str) -> Path:
    p = Path(rel_or_abs)
    target = p if p.is_absolute() else (project_root / p)
    if not target.exists():
        matches = list(project_root.glob(f"**/{p.name}"))
        if matches:
            target = matches[0]
    assert target.exists(), f"Config file not found: {rel_or_abs}"
    return target

def sync_folder(src: Path, dst: Path):
    if not src.exists() or src.resolve() == dst.resolve():
        return
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst, dirs_exist_ok=True)

def run_module_cmd(module_name: str, cfg_file: Path):
    cmd = [sys.executable, "-m", module_name, "--config", str(cfg_file)]
    print("Starting execution:")
    print(" ".join(cmd))
    print("-" * 70)
    proc = subprocess.Popen(
        cmd,
        cwd=str(project_root),
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    for line in proc.stdout:
        print(line, end="", flush=True)
    proc.wait()
    if proc.returncode != 0:
        raise RuntimeError(f"{module_name} failed for {cfg_file} with exit code {proc.returncode}")

# Restore cached lora_features from Google Drive if available
gdrive_lora_cache = GDRIVE_RUNS / "cache" / "lora_features"
local_lora_cache = project_root / "runs" / "cache" / "lora_features"
if gdrive_lora_cache.exists():
    print(f"Restoring cached LoRA features from Drive ({gdrive_lora_cache})...")
    sync_folder(gdrive_lora_cache, local_lora_cache)

print(f"Queued {len(CONFIG_PATHS)} experiment(s) to run:")
for i, cp in enumerate(CONFIG_PATHS, 1):
    print(f"  [{i}] {cp}")
print("=" * 70)

for idx, config_path in enumerate(CONFIG_PATHS, 1):
    print("\n" + "=" * 70)
    print(f"[{idx}/{len(CONFIG_PATHS)}] Starting Experiment: {config_path}")
    print("=" * 70)

    cfg_target = resolve_config(config_path)
    cfg_data = json.loads(cfg_target.read_text(encoding="utf-8"))

    # Check if this config depends on a LoRA run (e.g. runs/vlm/v10.3)
    fcfg = cfg_data.get("features", {"source": "frozen"})
    if fcfg.get("source") == "lora_run":
        lora_run_rel = Path(fcfg["run"])
        local_lora_dir = project_root / lora_run_rel
        gdrive_lora_dir = GDRIVE_RUNS / Path(*lora_run_rel.parts[1:]) if lora_run_rel.parts[0] == "runs" else (GDRIVE_RUNS / lora_run_rel)

        # 1. Try restoring from Google Drive if present
        if not (local_lora_dir / "lora_image_encoder.pt").exists() and (gdrive_lora_dir / "lora_image_encoder.pt").exists():
            print(f"Restoring prerequisite LoRA run from Drive: {gdrive_lora_dir} -> {local_lora_dir}")
            sync_folder(gdrive_lora_dir, local_lora_dir)

        # 2. If still missing, train the prerequisite LoRA config first
        if not (local_lora_dir / "lora_image_encoder.pt").exists():
            prereq_cfg_str = PREREQ_LORA_CONFIGS.get(lora_run_rel.as_posix())
            assert prereq_cfg_str is not None, f"Missing prerequisite LoRA run {lora_run_rel} and no config mapped in PREREQ_LORA_CONFIGS"
            prereq_cfg_path = resolve_config(prereq_cfg_str)
            print(f"Prerequisite LoRA run ({lora_run_rel}) not found. Training {prereq_cfg_str} first...")
            run_module_cmd("train.train_lora", prereq_cfg_path)
            sync_folder(local_lora_dir, gdrive_lora_dir)
            print(f"Synced prerequisite LoRA run to Drive: {gdrive_lora_dir}")

    METHOD_TO_MODULE = {
        "clip_zeroshot": "train.zeroshot",
        "clip_prompt_lora": "train.train_lora",
        "clip_feature_gen": "train.train_feature_gen",
    }
    module_name = METHOD_TO_MODULE.get(cfg_data.get("method"))
    assert module_name is not None, f"Unsupported method {cfg_data.get('method')!r} in {config_path}"
    t0 = time.time()
    run_module_cmd(module_name, cfg_target)

    # Sync run outputs and LoRA feature cache to Google Drive
    out_rel = Path(cfg_data["output_dir"])
    local_out_dir = project_root / out_rel
    gdrive_out_dir = GDRIVE_RUNS / Path(*out_rel.parts[1:]) if out_rel.parts[0] == "runs" else (GDRIVE_RUNS / out_rel)
    sync_folder(local_out_dir, gdrive_out_dir)
    if local_lora_cache.exists():
        sync_folder(local_lora_cache, gdrive_lora_cache)

    elapsed = (time.time() - t0) / 60.0
    print(f"\nCompleted [{idx}/{len(CONFIG_PATHS)}] {config_path} in {elapsed:.1f} minutes!")
    print(f"Synced outputs to Google Drive: {gdrive_out_dir}")

if globals().get("RUN_SIGLIP_ANCHORED_GAN", False):
    siglip_script = project_root / "Disease_Transfer_Experiments/clip_anchored_gan_v1/siglip_disease_only_official.py"
    for label, extra_args, out_sub in [
        ("SigLIP Untuned + Anchored GAN", ["--no-lora"], "official_siglip_untuned"),
        ("SigLIP LoRA + Anchored GAN", [], "official_siglip_lora"),
    ]:
        print("\n" + "=" * 70)
        print(f"Starting Two-Expert Experiment: {label}")
        print("=" * 70)
        cmd = [sys.executable, "-u", str(siglip_script), *extra_args]
        print(" ".join(cmd))
        proc = subprocess.Popen(cmd, cwd=str(project_root), env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        for line in proc.stdout:
            print(line, end="", flush=True)
        proc.wait()
        if proc.returncode != 0:
            raise RuntimeError(f"{label} failed with exit code {proc.returncode}")
        local_gan_out = project_root / "Disease_Transfer_Experiments/clip_anchored_gan_v1/clip_disease_only" / out_sub
        gdrive_gan_out = GDRIVE_RUNS / "clip_anchored_gan_v1" / out_sub
        sync_folder(local_gan_out, gdrive_gan_out)
        print(f"Synced {label} outputs to Google Drive: {gdrive_gan_out}")

print("\nAll queued experiments finished successfully and outputs are synced to Google Drive!")


Project root located at: /content
Restoring cached LoRA features from Drive (/content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/cache/lora_features)...
Queued 0 experiment(s) to run:

Starting Two-Expert Experiment: SigLIP Untuned + Anchored GAN
/usr/bin/python3 -u /content/Disease_Transfer_Experiments/clip_anchored_gan_v1/siglip_disease_only_official.py --no-lora
Using untuned frozen SigLIP (ViT-B-16-SigLIP-256) without LoRA
Fitting the one-epoch disease head with crop-anchored GAN vectors
disease_plus_gan_epoch=1 loss=1.6175 train_acc=0.6273
Traceback (most recent call last):
  File "/content/Disease_Transfer_Experiments/clip_anchored_gan_v1/siglip_disease_only_official.py", line 214, in <module>
    main(args.no_lora, args.disease_only_lora, args.gamma, args.lora_epochs)
    ~~~~^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/content/Disease_Transfer_Experiments/clip_anchored_gan_v1/siglip_disease_only_official.py", line 150, in main
    raise Fil

RuntimeError: SigLIP Untuned + Anchored GAN failed with exit code 1

In [45]:
# Step 6: Summarize All Configurations and Results (Uncalibrated & Calibrated)
import json
import numpy as np
import pandas as pd
from pathlib import Path
from IPython.display import display
from dpd.data import pair_table, read_metadata
from dpd.evaluate import evaluate

pairs_df = pair_table(read_metadata())

# Check queued configs plus all v11 CLIP-Anchored GAN configs if present
all_candidate_configs = list(CONFIG_PATHS)
for extra_cfg in [
    "config/vlm/v14_clip_siglip_zeroshot_config.json",
    "config/vlm/v14.1_clip_siglip_lora_config.json",
    "config/vlm/v11.4_lora_compositional_config.json",
    "config/vlm/v11.3_lora_features_config.json",
    "config/vlm/v11.1_compositional_condition_config.json",
    "config/vlm/v11_feature_gen_config.json",
]:
    if extra_cfg not in all_candidate_configs:
        all_candidate_configs.append(extra_cfg)

rows = []
for config_path in all_candidate_configs:
    cfg_file = Path(config_path) if Path(config_path).is_absolute() else (project_root / config_path)
    if not cfg_file.exists():
        continue
    cfg = json.loads(cfg_file.read_text(encoding="utf-8"))
    run_name = f"{cfg.get('version', cfg_file.stem)}_{cfg.get('title', '')}".rstrip("_")
    out_rel = Path(cfg["output_dir"])
    local_out_dir = project_root / out_rel
    gdrive_out_dir = GDRIVE_RUNS / Path(*out_rel.parts[1:]) if out_rel.parts[0] == "runs" else (GDRIVE_RUNS / out_rel)
    out_dir = gdrive_out_dir if (gdrive_out_dir / "metrics.json").exists() else local_out_dir

    metrics_file = out_dir / "metrics.json"
    scores_file = out_dir / "scores.npz"
    if not metrics_file.exists():
        if config_path in CONFIG_PATHS:
            rows.append({
                "experiment": run_name,
                "eval_mode": "missing",
                "epoch": None,
                "gamma": None,
                "val_hm_mean": None,
                "hm_top1": None,
                "hm_top5": None,
                "seen_top1": None,
                "seen_crop": None,
                "seen_disease": None,
                "unseen_top1": None,
                "unseen_crop": None,
                "unseen_disease": None,
                "diag_unseen_only": None,
                "config": config_path,
            })
        continue

    m_cal = json.loads(metrics_file.read_text(encoding="utf-8"))
    val_hm_mean = m_cal.get("val", {}).get("val_hm_mean")
    epoch = m_cal.get("selected_epoch")
    gamma_cal = float(m_cal.get("gamma", 0.0))

    candidates = []
    # Recompute full uncalibrated metrics from scores.npz if available
    if scores_file.exists():
        npz = np.load(scores_file)
        m_uncal = evaluate(
            npz["test_seen_scores"],
            npz["test_seen_labels"],
            npz["test_unseen_scores"],
            npz["test_unseen_labels"],
            pairs_df,
            0.0,
            tuple(cfg.get("evaluation", {}).get("top_k", [1, 5])),
        )
        candidates.append(("Uncalibrated (gamma=0.0)", 0.0, m_uncal))

    candidates.append((f"Calibrated (gamma={gamma_cal} applied)", gamma_cal, m_cal))

    for mode_label, g_val, m in candidates:
        ts = m.get("test_seen", {})
        tu = m.get("test_unseen", {})
        rows.append({
            "experiment": run_name,
            "eval_mode": mode_label,
            "epoch": epoch,
            "gamma": g_val,
            "val_hm_mean": val_hm_mean,
            "hm_top1": m.get("harmonic_mean_top1"),
            "hm_top5": m.get("harmonic_mean_top5"),
            "seen_top1": ts.get("top1"),
            "seen_crop": ts.get("crop_top1"),
            "seen_disease": ts.get("disease_top1"),
            "unseen_top1": tu.get("top1"),
            "unseen_crop": tu.get("crop_top1"),
            "unseen_disease": tu.get("disease_top1"),
            "diag_unseen_only": m.get("diagnostic_unseen_only_top1"),
            "config": config_path,
        })

df_summary = pd.DataFrame(rows)
pct_cols = [
    "val_hm_mean", "hm_top1", "hm_top5",
    "seen_top1", "seen_crop", "seen_disease",
    "unseen_top1", "unseen_crop", "unseen_disease", "diag_unseen_only",
]
df_display = df_summary.copy()
for col in pct_cols:
    if col in df_display.columns:
        df_display[col] = df_display[col].map(lambda x: f"{x:.2%}" if pd.notna(x) else "-")

print("================ CLIP-Anchored GAN Summary ================")
display(df_display)

summary_csv = GDRIVE_RUNS / "vlm" / "summary_clip_anchored_gan.csv"
summary_csv.parent.mkdir(parents=True, exist_ok=True)
df_summary.to_csv(summary_csv, index=False)
print(f"Saved summary CSV to: {summary_csv}")

gan_rows = []
for label, out_sub in [("SigLIP Untuned + Anchored GAN", "official_siglip_untuned"), ("SigLIP LoRA + Anchored GAN", "official_siglip_lora")]:
    for cand in [project_root / "Disease_Transfer_Experiments/clip_anchored_gan_v1/clip_disease_only" / out_sub / "results.json", GDRIVE_RUNS / "clip_anchored_gan_v1" / out_sub / "results.json"]:
        if cand.exists():
            res = json.loads(cand.read_text(encoding="utf-8"))
            for g_key, m in res.get("official", {}).items():
                gan_rows.append({"experiment": label, "gamma": g_key, "hm_top1": f"{m['h']:.2%}", "seen_top1": f"{m['seen']:.2%}", "unseen_top1": f"{m['unseen']:.2%}"})
            break
if gan_rows:
    print("\n================ SigLIP Anchored GAN Two-Expert Summary ================")
    display(pd.DataFrame(gan_rows))


================ CLIP-Anchored GAN Summary ================


,experiment,eval_mode,epoch,gamma,val_hm_mean,hm_top1,hm_top5,seen_top1,seen_crop,seen_disease,unseen_top1,unseen_crop,unseen_disease,diag_unseen_only,config
0,v14_siglip_zeroshot,Uncalibrated (gamma=0.0),NaN,0.0000,-,14.25%,46.06%,18.65%,65.70%,21.80%,11.53%,52.28%,14.57%,75.21%,config/vlm/v14_clip_siglip_zeroshot_config.json
1,v14_siglip_zeroshot,Calibrated (gamma=0.0 applied),NaN,0.0000,-,14.25%,46.06%,18.65%,65.70%,21.80%,11.53%,52.28%,14.57%,75.21%,config/vlm/v14_clip_siglip_zeroshot_config.json
2,v14.1_siglip_lora_text_anchor,Uncalibrated (gamma=0.0),15.0,0.0000,59.73%,29.08%,70.04%,70.75%,92.90%,72.45%,18.30%,81.91%,21.55%,92.47%,config/vlm/v14.1_clip_siglip_lora_config.json
3,v14.1_siglip_lora_text_anchor,Calibrated (gamma=0.0575 applied),15.0,0.0575,59.73%,69.23%,93.21%,61.65%,87.40%,65.75%,78.94%,92.13%,78.94%,92.47%,config/vlm/v14.1_clip_siglip_lora_config.json
4,v11.4_lora_compositional,Uncalibrated (gamma=0.0),4.0,0.0000,60.80%,33.56%,75.31%,69.75%,91.05%,72.05%,22.10%,76.31%,26.52%,81.98%,config/vlm/v11.4_lora_compositional_config.json
5,v11.4_lora_compositional,Calibrated (gamma=0.0525 applied),4.0,0.0525,60.80%,66.85%,90.40%,62.00%,84.00%,65.25%,72.51%,84.74%,72.58%,81.98%,config/vlm/v11.4_lora_compositional_config.json


Saved summary CSV to: /content/drive/MyDrive/Nat_AML_DPD_group_assignment/runs/vlm/summary_clip_anchored_gan.csv
